In [ ]:
"""
@author:norman lu and rodrigo pena
"""

import numpy as np
import matplotlib.pyplot as plt
#plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "DejaVu Sans"
import pandas as pd
import time
from scipy import signal
from scipy.ndimage import gaussian_filter
import seaborn as sns
import random
import pylab
pylab.rcParams['savefig.dpi'] = 120
#matplotlib inline

In [2]:
""" Parameters simulation """
dt = 0.1
tf = 2000
N = round(tf/dt)

In [3]:
"""stim functions"""

Ipoisson = lambda r: 1*(np.random.rand()<r*10**-3*dt)
Izap = lambda i, Fzap, A: A*np.sin(2*np.pi*Fzap*10**-3*i*dt + 3*np.pi/2)

def rect(T):
    """create a centered rectangular pulse of width $T"""
    return lambda t: (-T/2 <= t) & (t < T/2)

def pulse_train(t, at, shape):
    """create a train of pulses over $t at times $at and shape $shape"""
    return np.sum(shape(t - at[:,np.newaxis]), axis=0)

In [4]:
class PYR:
    """The PYR model"""
    nid=0 #default neuron id
    sendspk=0
    
    Cm=1
    gL=0.25
    g1=0.25
    tau1=100     
    
    vth=4
    vr=-1
    
    gsyn=1
    tausyn=5
    Esyn=5
    countdelay=0
    
    gext = 0.3 ### 0.3 is default
    tauext=5
    Eext=5
 
    spktimes = []
    
    def __init__(self,nid=1,gL=0.25,g1=0.25,tau1=100,Esyn=5,gsyn=1.5,tausyn=5):
        self.nid = nid
        self.v = 0
        self.w = 0
        self.s = 0
        self.sext=0
        self.gL = gL
        self.g1 = g1
        self.tau1 = tau1
        self.Esyn=Esyn
        self.gsyn=gsyn
        self.tausyn=tausyn
        self.spktimes = []

    def _UpdateKs(self,v,w,s,sext,stim):
        Iampa = self.gsyn*s * (v-self.Esyn) 
        Iext = self.gext*sext * (v-self.Eext) 
        Itotal = stim - self.gL*v - self.g1*w - Iampa - Iext
        kv = Itotal / (self.Cm)
        kw = (v-w) / self.tau1
        ks = (-s/self.tausyn)
        ksext = -sext/self.tauext
        return kv, kw, ks, ksext

    def _Updatev(self,stim,dt,time):
        if(self.countdelay>0):
            self.countdelay-=dt
        if(self.countdelay<0):
            self.countdelay=0
            self.s += 1
        k1v, k1w, k1s, k1sext = self._UpdateKs(self.v,self.w,self.s,self.sext,stim)
        a_v= self.v + k1v*dt/2
        a_w= self.w + k1w*dt/2
        a_s= self.s + k1s*dt/2
        a_sext= self.sext + k1sext*dt/2
        k2v, k2w, k2s, k2sext = self._UpdateKs(a_v,a_w,a_s,a_sext,stim)
        a_v= self.v + k2v*dt/2
        a_w= self.w + k2w*dt/2
        a_s= self.s + k2s*dt/2
        a_sext= self.sext + k2sext*dt/2
        k3v, k3w, k3s, k3sext = self._UpdateKs(a_v,a_w,a_s,a_sext,stim)
        a_v= self.v + k3v*dt
        a_w= self.w + k3w*dt
        a_s= self.s + k3s*dt
        a_sext= self.sext + k3sext*dt
        k4v, k4w, k4s, k4sext = self._UpdateKs(a_v,a_w,a_s,a_sext,stim)
        self.v += dt*(k1v+2*k2v+2*k3v+k4v)/6 #0.3*np.random.rand()*np.sqrt(dt) + 
        self.w += dt*(k1w+2*k2w+2*k3w+k4w)/6
        self.s += dt*(k1s+2*k2s+2*k3s+k4s)/6
        self.sext += dt*(k1sext+2*k2sext+2*k3sext+k4sext)/6
        self.sendspk=0
        if(self.v>self.vth):
            self.v=self.vr
            self.spktimes.append(time)
            self.sendspk=1
            
    def add_pre_spike(self,delay):
        self.countdelay = delay 
        self.sendspk=0
        
    def delay_count(self,delay):
        self.countdelay = delay 
        
    def add_external(self):
        self.sext += 1
        
    def eval(self,stim,dt,time):  
        self._Updatev(stim,dt,time)

In [ ]:
#!/usr/bin/env python3
# ------------------------------------------------------------
#   E/I synaptic-current monitor
# ------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

# ------------------------------------------------------------
# helper: Poisson spike generator
# ------------------------------------------------------------
def Ipoisson(rate_hz):
    """Return True with probability rate*dt (dt set globally, ms)."""
    return np.random.rand() < rate_hz * 1e-3 * dt


def make_inhom_poisson(rate_hz, tf, dt):
    """Constant-rate profile (could be replaced by anything)."""
    return rate_hz * np.ones(int(tf / dt))


# ------------------------------------------------------------
# single conductance-based LIF neuron (minimal implementation)
# ------------------------------------------------------------
class PYR:
    # (class-wide defaults; can be overridden in __init__)
    Cm = 200.0     # pF
    gL = 10.0      # nS
    EL = -65.0     # mV
    Vth = -50.0    # mV
    Vreset = -60.0 # mV
    tref = 2.0     # ms (absolute refractory)

    def __init__(self, nid, Esyn=0.0, tausyn=6.0, gsyn=2.0, g1=0, **kw):
        self.nid = nid
        self.Esyn = Esyn          # >-40 mV ≙ excitatory, else inhibitory
        self.tausyn = tausyn
        self.gmax = gsyn          # nS added per incoming spike
        # allow per-cell parameter overrides
        for k, v in kw.items():
            setattr(self, k, v)

        # dynamic state
        self.v = self.Vreset      # membrane potential (mV)
        self.reftime = 0.0        # remaining refractory time (ms)
        self.gE = 0.0             # excitatory conductance (nS)
        self.gI = 0.0             # inhibitory  conductance (nS)

        # variables that will be recorded each dt
        self.IsynE = 0.0          # instant Exc current  (nA)
        self.IsynI = 0.0          # instant Inh current  (nA)

        self.sendspk = 0          # 1 → spike was emitted **last** step
        self.spktimes = []        # list of spike times (ms)

    # ----------------------------------------------------------
    # external Poisson spike (always treated as excitatory)
    # ----------------------------------------------------------
    def add_external(self):
        self.gE += self.gmax

    # presynaptic partner spike (delivered after delay)
    def add_pre_spike(self, delay):
        if self.Esyn >= -40:      # excitatory cell
            self.gE += self.gmax
        else:                     # inhibitory cell
            self.gI += self.gmax

    # ----------------------------------------------------------
    # one Euler step
    # ----------------------------------------------------------
    def eval(self, Iext, dt, t):
        # conductance exponential decay
        self.gE -= dt * self.gE / self.tausyn
        self.gI -= dt * self.gI / self.tausyn

        # synaptic currents (positive = outward)
        self.IsynE = self.gE * (self.v - 0.0)         # Erev-E =   0 mV
        self.IsynI = self.gI * (self.v - (-75.0))     # Erev-I = −75 mV
        Isyn = self.IsynE + self.IsynI                # total (nA)

        # refractory?
        if self.reftime > 0:
            self.reftime -= dt
            self.v = self.Vreset
            self.sendspk = 0
            return

        # membrane integration (explicit Euler)
        dv = dt / self.Cm * (-self.gL * (self.v - self.EL) - Isyn + Iext)
        self.v += dv

        # spike threshold
        self.sendspk = 0
        if self.v >= self.Vth:
            self.v = self.Vreset
            self.reftime = self.tref
            self.sendspk = 1
            self.spktimes.append(t)


# ------------------------------------------------------------
# network / simulation parameters
# ------------------------------------------------------------
Nneuron = 25     # first population: 20 E + 5 I
Nex     = 20
delay   = 1.5    # ms axonal delay
tf      = 2000   # simulation length (ms)
dt      = 0.1    # integration step (ms)

# constant Poisson drive (8 Hz)
sig  = make_inhom_poisson(8.0, tf, dt)   # to first population
sig2 = make_inhom_poisson(8.0, tf, dt)   # to second population

asc  = np.linspace(0.5, 3.0, 10)    # upward branch
desc = asc[-2::-1]                  # same values, reverse order, skip 3.0
input_strengths = np.concatenate((asc, desc))
firing_rates_pop1, firing_rates_pop2 = [], []
mean_Iexc, mean_Iinh = [], []               # time-averaged currents

# ------------------------------------------------------------
# outer loop: vary input strength
# ------------------------------------------------------------
for in_str in input_strengths:
    # --------------------------------------------------------
    # build network
    # --------------------------------------------------------
    neuron_list, syn_list = [], []

    # first 25 neurons
    for i in range(Nneuron):
        if i < Nex:      # excitatory cells
            neuron_list.append(PYR(i, Esyn=0.0,  tausyn=6, gsyn=2))
            syn_list.append(np.arange(Nex, Nneuron))  # → all 5 interneurons
        else:            # inhibitory cells
            neuron_list.append(PYR(i, Esyn=-75, tausyn=6,
                                   gsyn=6 * in_str))
            syn_list.append(np.r_[np.arange(0, Nex),      # all 20 E
                                  np.arange(Nneuron, Nneuron+Nex)])
    # second population: another 20 excitatory cells with variable gsyn
    for i in range(Nex):
        idx = Nneuron + i
        neuron_list.append(PYR(idx, Esyn=0.0, tausyn=6,
                               gsyn=4 * in_str))
        syn_list.append(np.arange(Nex, Nneuron))          # → 5 interneurons

    # storage
    nt = int(tf / dt)
    v  = np.empty((len(neuron_list), nt))
    I_E = np.zeros(nt)          # population-averaged excitatory current
    I_I = np.zeros(nt)          # population-averaged inhibitory  current

    # --------------------------------------------------------
    # simulation loop
    # --------------------------------------------------------
    tic = time.time()
    for i in range(nt - 1):
        exc_sum, inh_sum = 0.0, 0.0

        for n, cell in enumerate(neuron_list):
            # external Poisson input
            if n < Nex and Ipoisson(sig[i]):
                cell.add_external()
            if n >= Nneuron and Ipoisson(sig2[i]):
                cell.add_external()

            # deliver outgoing spikes
            if cell.sendspk == 1 and n < len(syn_list):
                for tgt in syn_list[n]:
                    if tgt < len(neuron_list):
                        neuron_list[tgt].add_pre_spike(delay)

            # membrane update
            cell.eval(0.0, dt, i * dt)
            v[n, i] = cell.v

            exc_sum += cell.IsynE
            inh_sum += cell.IsynI

        I_E[i] = exc_sum / len(neuron_list)
        I_I[i] = inh_sum / len(neuron_list)
    toc = time.time()
    print(f'input={in_str:4.2f} : simulated in {toc - tic:4.1f} s')

    # --------------------------------------------------------
    # spike raster → firing rates
    # --------------------------------------------------------
    df = pd.DataFrame({'nid': [], 'spktime': []})
    for n, cell in enumerate(neuron_list):
        if cell.spktimes:
            df = pd.concat([df, pd.DataFrame({'nid': n,
                                              'spktime': cell.spktimes})])
    firing_rates_pop1.append(len(df[df.nid <  Nex])     / (tf * Nex * 1e-3))
    firing_rates_pop2.append(len(df[df.nid >= Nneuron]) / (tf * Nex * 1e-3))

    mean_Iexc.append(np.mean(I_E)+5)
    mean_Iinh.append(np.mean(I_I))

    # --------------------------------------------------------
    # plot currents for this input strength
    # --------------------------------------------------------
    tvec = np.arange(0, tf, dt)
    plt.figure(figsize=(7, 4))
    plt.plot(tvec, I_E, 'r', label='Excitatory current')
    plt.plot(tvec, I_I, 'b', label='Inhibitory current')
    plt.xlabel('Time (ms)')
    plt.ylabel('Population-mean synaptic current (nA)')
    plt.title(f'Synaptic currents (input = {in_str:.2f})')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'syn_currents_{in_str:.2f}.png', dpi=300)
    plt.close()

# ------------------------------------------------------------
# final summary figures
# ------------------------------------------------------------
plt.figure(figsize=(6, 4))
plt.plot(input_strengths, firing_rates_pop1, 'o-b', label='Dystonic Synergy')
plt.plot(input_strengths, firing_rates_pop2, 's-r', label='Functional Synergy')
plt.xlabel('Input strength')
plt.ylabel('Firing rate (Hz)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('further_unbalance.png', dpi=300)

plt.figure(figsize=(6, 4))
plt.plot(input_strengths, mean_Iexc, 'o-r', label='I_E')
plt.plot(input_strengths, mean_Iinh, 's-b', label='I_I')
plt.xlabel('Input strength')
plt.ylabel('Time-averaged synaptic current (nA)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('mean_syn_currents.png', dpi=300)

print('All plots written to disk – done.')

In [ ]:
# ------------------------------------------------------------
# final summary figures
# ------------------------------------------------------------

# --- put everything in ascending x-order --------------------
idx = np.argsort(input_strengths)          # permutation indices
xs        = input_strengths[idx]
fr1       = np.asarray(firing_rates_pop1)[idx]
fr2       = np.asarray(firing_rates_pop2)[idx]
mean_exc  = np.asarray(mean_Iexc)[idx]
mean_inh  = np.asarray(mean_Iinh)[idx]

# ---- 1. firing-rate figure ---------------------------------
plt.figure(figsize=(6, 4))
plt.subplot(211)
plt.plot(xs, fr1, 'o-',  label='Functional Synergy')
plt.plot(xs, fr2, 's-',  label='Dystonic Synergy')
plt.xlabel('Input strength')
plt.ylabel('Firing rate (Hz)')
plt.grid(True)
plt.legend()
plt.tight_layout()
# plt.savefig('further_unbalance.png', dpi=300)

# ---- 2. mean current figure --------------------------------
plt.subplot(212)
plt.plot(xs, mean_exc, 'o-', color='r', label='I_E')
plt.plot(xs, mean_inh, 's-', color='b', label='I_I')
plt.xlabel('Input strength')
plt.ylabel('Time-averaged synaptic current (nA)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('mean_syn_currents.png', dpi=300)